In [1]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to ~/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [13]:
import sys

!{sys.executable} -m pip install -q nltk
# !{sys.executable} -m pip install numpy
!{sys.executable} -m pip install "numpy<2.0" --force-reinstall



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 28.8 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5


In [5]:
!pip install numpy pandas matplotlib seaborn scikit-learn

In [4]:
!pip uninstall -y transformers bert-score torch safetensors
!pip install "torch==2.2.2" "transformers==4.36.2" "bert-score==0.3.13" "safetensors"

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
Found existing installation: bert-score 0.3.13
Uninstalling bert-score-0.3.13:
  Successfully uninstalled bert-score-0.3.13
Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: safetensors 0.7.0
Uninstalling safetensors-0.7.0:
  Successfully uninstalled safetensors-0.7.0
  Using cached torch-2.2.2-cp311-none-macosx_10_9_x86_64.whl.metadata (25 kB)
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_10_12_x86_64.whl.metadata (4.1 kB)
Using cached torch-2.2.2-cp311-none-macosx_10_9_x86_64.whl (150.8 MB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
Using cached bert_score-0.3.13-py3-none-any.whl (61 kB)
Using cached safetensors-0.7.0-

In [10]:
import sys
print(sys.executable)


/usr/local/opt/python@3.11/bin/python3.11


In [1]:
import torch
import numpy as np

print("torch ok:", torch.__version__)
print("numpy ok:", np.__version__)

# this will fail if torch still can't see numpy
a = torch.tensor([1], dtype=torch.float32).numpy()
print(a)


torch ok: 2.2.2
numpy ok: 1.26.4
[1.]


In [2]:
import transformers, torch, bert_score

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("bert-score:", bert_score.__version__)

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


torch: 2.2.2
transformers: 4.36.2
bert-score: 0.3.12


In [3]:

# BERTScore settings
BERT_MODEL = "roberta-large"
BERT_LANG = "en"
BERT_USE_IDF = True
BERT_RESCALE_BASELINE = False

LOG_DEBUG = True
CAND_SNIP = 300


# ---- HELPERS ----
import os, re, glob, warnings
import numpy as np
import pandas as pd
from bert_score import score as bertscore


In [16]:
import re

def read_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

In [182]:
import re

def load_three_sections(path: str):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    # Split on the marker AND remove the marker from the extracted text
    parts = re.split(r"=== COMPLIANCE ASSESSMENT ===", raw)

    # Remove empty entries and whitespace
    parts = [p.strip() for p in parts if p.strip()]

    if len(parts) < 3:
        raise ValueError(f"Expected 3 sections, found {len(parts)}.")

    # Now A, B, C have ONLY text, no markers
    return parts[0], parts[1], parts[2]
A, B, C = load_three_sections("../evaluations/Experiment3/Mistral/Assessment_gnb_cuup_sa")  # or your real path

In [183]:
print("A length:", len(A))
print("B length:", len(B))
print("C length:", len(C))

print("\nA preview:", repr(A[:200]))
print("\nB preview:", repr(B[:200]))
print("\nC preview:", repr(C[:200]))


A length: 6670
B length: 5916
C length: 4340

A preview: 'Here is the **Detection Phase** summary of security-relevant fields in the provided `gnb-cuup.sa.f1.conf` configuration:\n\n### **Security-Relevant Fields Detected**\n1. **F1 Interface Configuration**\n  '

B preview: '### **Detection Phase**\nThe following security-relevant fields were identified in the `gnb-cuup.sa.f1.conf` configuration file:\n\n1. **PLMN Configuration**\n   - `mcc = 208`\n   - `mnc = 99`\n   - `mnc_le'

C preview: '### **Detection Phase**\nThe following security-relevant fields were identified in the `gnb-cuup.sa.f1.conf` configuration file:\n\n1. **PLMN Configuration**\n   - `mcc = 208`\n   - `mnc = 99`\n   - `mnc_le'


In [184]:

BERT_MODEL = "roberta-large"   # or whatever you used
BERT_LANG = "en"
BERT_USE_IDF = False
BERT_RESCALE_BASELINE = False

def bert_F1(a: str, b: str) -> float:
    P, R, F1 = bertscore(
        [a], [b],
        model_type=BERT_MODEL,
        lang=BERT_LANG,
        idf=BERT_USE_IDF,
        rescale_with_baseline=BERT_RESCALE_BASELINE,
    )
    return float(F1[0])

sAB = bert_F1(A, B)
sAC = bert_F1(A, C)
sBC = bert_F1(B, C)

print("A vs B:", sAB)
print("A vs C:", sAC)
print("B vs C:", sBC)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


A vs B: 0.9403330087661743
A vs C: 0.9377301335334778
B vs C: 0.9773290753364563


In [185]:
overall = (sAB + sAC + sBC) / 3
print("Overall BERTScore F1:", overall)

Overall BERTScore F1: 0.9517974058787028
